# Marimo 101: SQL on OMOP, the Jupyter version

This is one of three versions of the same lesson, all built on the same OMOP data and the same queries:

| Version | File | How it runs |
|---|---|---|
| **Quarto deck** | `sql-basics.qmd` | reveal.js slides; marimo cells run as *islands* in the browser |
| **Jupyter notebook** *(this one)* | `sql-basics-jupyter.ipynb` | an IPython kernel; widgets use ipywidgets callbacks; you re-run cells by hand |
| **marimo notebook** | `sql-basics-marimo.py` | a reactive Python file you can open locally, on molab, or in the browser (WASM) |

Look for the 🟠 **Jupyter note** callouts. They show where Jupyter needs extra code, or behaves differently from marimo.

**Run it**

```bash
uv run --with jupyterlab --with ipywidgets --with anywidget --with duckdb \
       --with pandas --with numpy --with altair jupyter lab sql-basics-jupyter.ipynb
```

Then **Run → Run All Cells**. Unlike marimo, the order you run cells in matters.

In [ ]:
from pathlib import Path
import base64, csv

import duckdb
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as w
from IPython.display import display, Markdown, HTML

## Learning SQL with OMOP

Every OMOP extract uses **the same tables and column names**, so a query written here can run against any OMOP dataset at any institution.

- Clinical facts live in *event tables*
- Their meaning lives in **`concept`**, the vocabulary table

Most of the queries we'll see **(1)** filter an event table, **(2)** join to `person`, and **(3)** join to `concept` to get readable names.

```mermaid
erDiagram
    PERSON ||--o{ VISIT_OCCURRENCE : has
    PERSON ||--o{ CONDITION_OCCURRENCE : has
    PERSON ||--o{ DRUG_EXPOSURE : has
    PERSON ||--o{ MEASUREMENT : has
    VISIT_OCCURRENCE ||--o{ CONDITION_OCCURRENCE : "recorded during"
    CONCEPT ||--o{ CONDITION_OCCURRENCE : "condition_concept_id"
    CONCEPT ||--o{ DRUG_EXPOSURE : "drug_concept_id"
    CONCEPT ||--o{ MEASUREMENT : "measurement_concept_id"
    PERSON {
        int person_id PK
        int gender_concept_id FK
        int year_of_birth
    }
    CONDITION_OCCURRENCE {
        int person_id FK
        int condition_concept_id FK
        date condition_start_date
    }
    CONCEPT {
        int concept_id PK
        string concept_name
        string domain_id
    }
```

## The dataset

- A synthetic OMOP CDM subset in **DuckDB**. If `omop.duckdb` sits next to this notebook it is opened read-only; otherwise (Colab, Binder) the cell rebuilds the same seeded tables in memory
- Concept IDs are real standard concepts (SNOMED, RxNorm, LOINC)
- People, dates and values are simulated

> 🟠 **Jupyter note:** Jupyter has no `mo.notebook_dir()`, so the path is relative to wherever the kernel was started.

In [ ]:
# ---------------------------------------------------------------
# Synthetic OMOP CDM v5.4 subset stored in omop.duckdb.
# Concept IDs are real standard OMOP concepts; people, dates and
# values are simulated. Rebuild with: uv run build_omop_db.py
# ---------------------------------------------------------------
DB_PATH = Path.cwd() / "omop.duckdb"
if DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH), read_only=True)
else:
    # Same seeded tables as build_omop_db.py, rebuilt in memory.
        _rng = np.random.default_rng(42)
        _N = 1200
        _t0 = pd.Timestamp("2016-01-01")
        _span = (pd.Timestamp("2024-12-31") - _t0).days

        _concept = pd.DataFrame(
            [
                (8507, "MALE", "Gender", "Gender"),
                (8532, "FEMALE", "Gender", "Gender"),
                (8527, "White", "Race", "Race"),
                (8516, "Black or African American", "Race", "Race"),
                (8515, "Asian", "Race", "Race"),
                (38003563, "Hispanic or Latino", "Ethnicity", "Ethnicity"),
                (38003564, "Not Hispanic or Latino", "Ethnicity", "Ethnicity"),
                (9201, "Inpatient Visit", "Visit", "Visit"),
                (9202, "Outpatient Visit", "Visit", "Visit"),
                (9203, "Emergency Room Visit", "Visit", "Visit"),
                (201826, "Type 2 diabetes mellitus", "Condition", "SNOMED"),
                (320128, "Essential hypertension", "Condition", "SNOMED"),
                (317576, "Coronary arteriosclerosis", "Condition", "SNOMED"),
                (255573, "Chronic obstructive lung disease", "Condition", "SNOMED"),
                (46271022, "Chronic kidney disease", "Condition", "SNOMED"),
                (433736, "Obesity", "Condition", "SNOMED"),
                (4329847, "Myocardial infarction", "Condition", "SNOMED"),
                (440383, "Depressive disorder", "Condition", "SNOMED"),
                (197320, "Acute renal failure syndrome", "Condition", "SNOMED"),
                (1503297, "metformin", "Drug", "RxNorm"),
                (1308216, "lisinopril", "Drug", "RxNorm"),
                (1545958, "atorvastatin", "Drug", "RxNorm"),
                (1112807, "aspirin", "Drug", "RxNorm"),
                (1307046, "metoprolol", "Drug", "RxNorm"),
                (1332418, "amlodipine", "Drug", "RxNorm"),
                (3004410, "Hemoglobin A1c/Hemoglobin.total in Blood", "Measurement", "LOINC"),
                (3016723, "Creatinine [Mass/volume] in Serum or Plasma", "Measurement", "LOINC"),
                (3004249, "Systolic blood pressure", "Measurement", "LOINC"),
            ],
            columns=["concept_id", "concept_name", "domain_id", "vocabulary_id"],
        )

        # person -------------------------------------------------------------
        _person = pd.DataFrame({
            "person_id": np.arange(1, _N + 1),
            "gender_concept_id": _rng.choice([8507, 8532], _N, p=[.48, .52]),
            "year_of_birth": _rng.integers(1935, 2006, _N),
            "month_of_birth": _rng.integers(1, 13, _N),
            "race_concept_id": _rng.choice([8527, 8516, 8515, 0], _N, p=[.6, .2, .08, .12]),
            "ethnicity_concept_id": _rng.choice([38003563, 38003564], _N, p=[.25, .75]),
        })
        _age = dict(zip(_person.person_id, 2024 - _person.year_of_birth))

        # visit_occurrence ---------------------------------------------------
        _nv = (_rng.poisson(4, _N) + 1).astype(np.intp)  # np.repeat needs intp (int32 in WASM)
        _vp = np.repeat(_person.person_id.values, _nv)
        _V = len(_vp)
        _vtype = _rng.choice([9201, 9202, 9203], _V, p=[.12, .75, .13])
        _vstart = _t0 + pd.to_timedelta(_rng.integers(0, _span, _V), unit="D")
        _los = np.where(_vtype == 9201, _rng.integers(1, 11, _V), 0)
        _visit = pd.DataFrame({
            "visit_occurrence_id": np.arange(1, _V + 1),
            "person_id": _vp,
            "visit_concept_id": _vtype,
            "visit_start_date": _vstart,
            "visit_end_date": _vstart + pd.to_timedelta(_los, unit="D"),
        })
        _by_person = {p: g for p, g in _visit.groupby("person_id")}

        # condition_occurrence ------------------------------------------------
        _base = {201826: .16, 320128: .28, 317576: .09, 255573: .07, 46271022: .06,
                 433736: .15, 4329847: .03, 440383: .12, 197320: .04}
        _crow = []
        for _pid, _g in _by_person.items():
            _scale = 0.35 + _age[_pid] / 55
            for _cid, _p in _base.items():
                if _rng.random() < min(_p * _scale, .85):
                    _v = _g.iloc[_rng.integers(0, len(_g))]
                    _crow.append((_pid, _cid, _v.visit_start_date, _v.visit_occurrence_id))
        _cond = pd.DataFrame(_crow, columns=["person_id", "condition_concept_id",
                                             "condition_start_date", "visit_occurrence_id"])
        _cond.insert(0, "condition_occurrence_id", np.arange(1, len(_cond) + 1))
        _has = _cond.groupby("person_id").condition_concept_id.apply(set).to_dict()

        # drug_exposure -------------------------------------------------------
        _drow = []
        for _pid, _g in _by_person.items():
            _cs = _has.get(_pid, set())
            if 201826 in _cs and _rng.random() < .72:
                _d0 = _cond[(_cond.person_id == _pid) & (_cond.condition_concept_id == 201826)].condition_start_date.iloc[0]
                _drow.append((_pid, 1503297, _d0 + pd.Timedelta(days=int(_rng.integers(0, 240)))))
            if 320128 in _cs:
                for _dc in (1308216, 1332418):
                    if _rng.random() < .45:
                        _drow.append((_pid, _dc, _g.visit_start_date.iloc[_rng.integers(0, len(_g))]))
            if _cs & {317576, 4329847}:
                for _dc in (1545958, 1112807, 1307046):
                    if _rng.random() < .6:
                        _drow.append((_pid, _dc, _g.visit_start_date.iloc[_rng.integers(0, len(_g))]))
        _drug = pd.DataFrame(_drow, columns=["person_id", "drug_concept_id", "drug_exposure_start_date"])
        _drug["days_supply"] = _rng.choice([30, 90], len(_drug))
        _drug["drug_exposure_end_date"] = _drug.drug_exposure_start_date + pd.to_timedelta(_drug.days_supply, unit="D")
        _drug.insert(0, "drug_exposure_id", np.arange(1, len(_drug) + 1))

        # measurement ---------------------------------------------------------
        _mrow = []
        for _v in _visit.itertuples():
            _cs = _has.get(_v.person_id, set())
            if _rng.random() < .35:
                _a1c = _rng.normal(7.9, 1.3) if 201826 in _cs else _rng.normal(5.4, .35)
                _mrow.append((_v.person_id, 3004410, _v.visit_start_date, round(max(_a1c, 4.0), 1), "%"))
            if _rng.random() < .6:
                _cr = _rng.lognormal(0, .2) * (1.9 if 197320 in _cs and _rng.random() < .5 else 1.0)
                _cr *= 1.35 if 46271022 in _cs else 1.0
                _mrow.append((_v.person_id, 3016723, _v.visit_start_date, round(_cr, 2), "mg/dL"))
            if _rng.random() < .8:
                _sbp = _rng.normal(141 if 320128 in _cs else 122, 13)
                _mrow.append((_v.person_id, 3004249, _v.visit_start_date, round(_sbp), "mmHg"))
        _meas = pd.DataFrame(_mrow, columns=["person_id", "measurement_concept_id",
                                             "measurement_date", "value_as_number", "unit_source_value"])
        _meas.insert(0, "measurement_id", np.arange(1, len(_meas) + 1))

        # load into DuckDB with proper DATE columns ---------------------------
        con = duckdb.connect()
        for _name, _df in {"person": _person, "visit_occurrence": _visit,
                           "condition_occurrence": _cond, "drug_exposure": _drug,
                           "measurement": _meas, "concept": _concept}.items():
            con.register("_tmp", _df)
            _cols = ", ".join(
                f"CAST({c} AS DATE) AS {c}" if c.endswith("_date") else c for c in _df.columns
            )
            con.execute(f"CREATE TABLE {_name} AS SELECT {_cols} FROM _tmp")
            con.unregister("_tmp")


def run(q):
    """Run SQL against the OMOP connection and return a pandas DataFrame."""
    return con.execute(q).df()


def show_sql(q):
    return Markdown(f"```sql\n{q.strip()}\n```")


CLAUSE = {"select": "#0021A5", "from": "#002657", "where": "#C63A0E",
          "join": "#6A2A60", "group": "#22884C", "having": "#7A5C00",
          "order": "#D32737", "window": "#343741",
          "ink": "#1A1A1A", "muted": "#5F6168"}


def styled(chart, w=520, h=300):
    return (chart.properties(width=w, height=h)
            .configure(font="Georgia")
            .configure_view(stroke=None)
            .configure_axis(labelColor=CLAUSE["ink"], titleColor=CLAUSE["muted"],
                            gridColor="#E6E7E6", domainColor="#C7C9C8",
                            labelFontSize=13, titleFontSize=13)
            .configure_legend(labelFontSize=13, titleFontSize=13))

print("Source:", DB_PATH.name if DB_PATH.exists() else "rebuilt in memory (seed 42)")

In [ ]:
_counts = run("""
    SELECT 'person' AS table_name, COUNT(*) AS n_rows FROM person
    UNION ALL SELECT 'visit_occurrence', COUNT(*) FROM visit_occurrence
    UNION ALL SELECT 'condition_occurrence', COUNT(*) FROM condition_occurrence
    UNION ALL SELECT 'drug_exposure', COUNT(*) FROM drug_exposure
    UNION ALL SELECT 'measurement', COUNT(*) FROM measurement
    UNION ALL SELECT 'concept', COUNT(*) FROM concept
""")
_bars = alt.Chart(_counts).mark_bar(color=CLAUSE["from"], cornerRadiusEnd=3).encode(
    x=alt.X("n_rows:Q", title="rows"),
    y=alt.Y("table_name:N", sort="-x", title=None),
    tooltip=["table_name", alt.Tooltip("n_rows:Q", format=",")],
)
_labels = _bars.mark_text(align="left", dx=4, color=CLAUSE["ink"]).encode(
    text=alt.Text("n_rows:Q", format=","))
styled(_bars + _labels, w=520, h=220)

## SQL query syntax

You write clauses in one order. The database evaluates them in another, which explains most beginner errors.

In [ ]:
HTML("""
<svg viewBox="0 0 1180 430" width="100%" role="img" aria-label="Written clause order versus logical evaluation order">
  <style>
    .lab { font: 700 20px 'JetBrains Mono', monospace; fill: #fff; }
    .hd  { font: 700 22px Georgia, serif; fill: #1A1A1A; }
    .note{ font: 17px Georgia, serif; fill: #5F6168; }
    .ln  { stroke: #9A9C9B; stroke-width: 2; fill: none; }
  </style>
  <text class="hd" x="40" y="30">How you write it</text>
  <text class="hd" x="640" y="30">How the database runs it</text>
  <g>
    <rect x="40"  y="50"  width="260" height="40" rx="5" fill="#0021A5"/><text class="lab" x="56" y="77">SELECT</text>
    <rect x="40"  y="97"  width="260" height="40" rx="5" fill="#002657"/><text class="lab" x="56" y="124">FROM</text>
    <rect x="40"  y="144" width="260" height="40" rx="5" fill="#6A2A60"/><text class="lab" x="56" y="171">JOIN … ON</text>
    <rect x="40"  y="191" width="260" height="40" rx="5" fill="#C63A0E"/><text class="lab" x="56" y="218">WHERE</text>
    <rect x="40"  y="238" width="260" height="40" rx="5" fill="#22884C"/><text class="lab" x="56" y="265">GROUP BY</text>
    <rect x="40"  y="285" width="260" height="40" rx="5" fill="#7A5C00"/><text class="lab" x="56" y="312">HAVING</text>
    <rect x="40"  y="332" width="260" height="40" rx="5" fill="#D32737"/><text class="lab" x="56" y="359">ORDER BY</text>
    <rect x="40"  y="379" width="260" height="40" rx="5" fill="#5F6168"/><text class="lab" x="56" y="406">LIMIT</text>
  </g>
  <g>
    <rect x="640" y="50"  width="260" height="40" rx="5" fill="#002657"/><text class="lab" x="656" y="77">1  FROM</text>
    <rect x="640" y="97"  width="260" height="40" rx="5" fill="#6A2A60"/><text class="lab" x="656" y="124">2  JOIN … ON</text>
    <rect x="640" y="144" width="260" height="40" rx="5" fill="#C63A0E"/><text class="lab" x="656" y="171">3  WHERE</text>
    <rect x="640" y="191" width="260" height="40" rx="5" fill="#22884C"/><text class="lab" x="656" y="218">4  GROUP BY</text>
    <rect x="640" y="238" width="260" height="40" rx="5" fill="#7A5C00"/><text class="lab" x="656" y="265">5  HAVING</text>
    <rect x="640" y="285" width="260" height="40" rx="5" fill="#0021A5"/><text class="lab" x="656" y="312">6  SELECT</text>
    <rect x="640" y="332" width="260" height="40" rx="5" fill="#D32737"/><text class="lab" x="656" y="359">7  ORDER BY</text>
    <rect x="640" y="379" width="260" height="40" rx="5" fill="#5F6168"/><text class="lab" x="656" y="406">8  LIMIT</text>
  </g>
  <path class="ln" d="M300,70 C470,70 470,305 640,305"/>
  <path class="ln" d="M300,117 C470,117 470,70 640,70"/>
  <path class="ln" d="M300,164 C470,164 470,117 640,117"/>
  <path class="ln" d="M300,211 C470,211 470,164 640,164"/>
  <path class="ln" d="M300,258 C470,258 470,211 640,211"/>
  <path class="ln" d="M300,305 C470,305 470,258 640,258"/>
  <path class="ln" d="M300,352 L640,352"/>
  <path class="ln" d="M300,399 L640,399"/>
  <text class="note" x="920" y="310">Aliases are born here,</text>
  <text class="note" x="920" y="332">so WHERE can’t see them</text>
</svg>
""")

## SELECT and FROM

- `FROM` names the table
- `SELECT` picks the columns you want back
- `LIMIT` keeps the preview small while you explore
- Avoid `SELECT *` in real extracts since OMOP tables are wide, and naming columns documents intent

> 🟠 **Jupyter note:** SQL is a Python string passed to `con.execute()`. There is no SQL cell type, no schema browser, and the result is only kept if you assign it. (Extensions such as `jupysql` add a `%%sql` magic.)

In [ ]:
person_preview = run("""
SELECT person_id,
       gender_concept_id,
       year_of_birth,
       race_concept_id
FROM person
LIMIT 6
""")
person_preview

## WHERE filters rows

Move the controls. The query text and the result both update.

- `BETWEEN` is inclusive on both ends
- Filtering on `gender_concept_id`, not a text label, is the OMOP way

> 🟠 **Jupyter note:** interactivity comes from **`ipywidgets.interact`**. The widgets and the function that uses them must live in the **same cell**, because a widget in one cell can't trigger a different cell. In marimo, any cell that reads `yob.value` reruns on its own.

In [ ]:
@w.interact(
    yob=w.IntRangeSlider(value=(1950, 1975), min=1935, max=2005, description="Year of birth",
                         continuous_update=False, layout=w.Layout(width="480px"),
                         style={"description_width": "initial"}),
    sex=w.Dropdown(options=[("Anyone", 0), ("Female (8532)", 8532), ("Male (8507)", 8507)],
                   value=0, description="Sex"),
)
def where_demo(yob, sex):
    sex_sql = "" if sex == 0 else f"\n  AND gender_concept_id = {sex}"
    where_sql = f"""
SELECT person_id, gender_concept_id, year_of_birth
FROM person
WHERE year_of_birth BETWEEN {yob[0]} AND {yob[1]}{sex_sql}
"""
    df = run(where_sql)
    display(show_sql(where_sql),
            Markdown(f"**{len(df):,}** of {run('SELECT COUNT(*) n FROM person').n[0]:,} people match"),
            df.head(5))

## ORDER BY and LIMIT

Without `ORDER BY`, row order is not guaranteed, so `LIMIT` alone returns an arbitrary sample.

In [ ]:
@w.interact(
    sort_col=w.Dropdown(options=["year_of_birth", "person_id", "month_of_birth"], description="Sort by"),
    sort_dir=w.ToggleButtons(options=["ASC", "DESC"], value="DESC", description="Direction"),
    top_n=w.BoundedIntText(value=8, min=1, max=25, description="LIMIT"),
)
def order_demo(sort_col, sort_dir, top_n):
    order_sql = f"""
SELECT person_id, year_of_birth, month_of_birth
FROM person
ORDER BY {sort_col} {sort_dir}
LIMIT {top_n}
"""
    display(show_sql(order_sql), run(order_sql))

## JOIN turns IDs into meaning

- Event tables store integers
- `concept` holds the names, joined with `ON co.condition_concept_id = c.concept_id`

In [ ]:
run("""
SELECT c.concept_name AS condition,
       COUNT(*)       AS n_records
FROM condition_occurrence AS co
JOIN concept AS c
  ON co.condition_concept_id = c.concept_id
GROUP BY c.concept_name
ORDER BY n_records DESC
LIMIT 5
""")

## INNER, LEFT and ANTI JOINS

> 🟠 **Jupyter note:** the toy diagram and the query both depend on the radio button, so they share one `interact` function. In marimo they are separate cells (and the diagram is its own variable, `join_toy`).

In [ ]:
def join_toy_svg(kind):
    # Toy example: 5 people, 4 condition rows. Highlight what survives the join.
    people = [1, 2, 3, 4, 5]
    conds = [(1, "HTN"), (1, "T2DM"), (3, "CKD"), (4, "HTN")]
    if kind == "INNER JOIN":
        rows = [(p, c) for p, c in conds]
    elif kind == "LEFT JOIN":
        rows = [(p, c) for p in people for q, c in conds if q == p] + \
               [(p, "NULL") for p in people if p not in {q for q, _ in conds}]
        rows.sort()
    else:
        rows = [(p, "NULL") for p in people if p not in {q for q, _ in conds}]
    kept = {p for p, _ in rows}
    svg = ['<svg viewBox="0 0 560 250" width="100%" style="max-width:560px" role="img" aria-label="Toy join result">',
           '<style>.t{font:16px "JetBrains Mono",monospace}.h{font:700 15px Georgia,serif;fill:#5F6168}</style>',
           '<text class="h" x="0" y="16">person</text><text class="h" x="130" y="16">condition</text>',
           '<text class="h" x="340" y="16">result</text>']
    for i, p in enumerate(people):
        on = p in kept
        svg.append(f'<rect x="0" y="{26+i*40}" width="90" height="32" rx="4" fill="{"#F3E9F1" if on else "#fff"}" stroke="{"#6A2A60" if on else "#C7C9C8"}"/>')
        svg.append(f'<text class="t" x="12" y="{48+i*40}" fill="{"#1A1A1A" if on else "#9A9C9B"}">id {p}</text>')
    for i, (p, c) in enumerate(conds):
        on = kind != "Anti join (LEFT + IS NULL)"
        svg.append(f'<rect x="130" y="{26+i*40}" width="150" height="32" rx="4" fill="#fff" stroke="{"#6A2A60" if on else "#C7C9C8"}"/>')
        svg.append(f'<text class="t" x="140" y="{48+i*40}" fill="{"#1A1A1A" if on else "#9A9C9B"}">{p} · {c}</text>')
    for i, (p, c) in enumerate(rows[:6]):
        svg.append(f'<rect x="340" y="{26+i*37}" width="210" height="30" rx="4" fill="#d17ac4"/>')
        svg.append(f'<text class="t" x="352" y="{47+i*37}" fill="#fff">{p} · {c}</text>')
    svg.append("</svg>")
    return HTML("".join(svg))


@w.interact(kind=w.RadioButtons(options=["INNER JOIN", "LEFT JOIN", "Anti join (LEFT + IS NULL)"],
                                description="Join type"))
def join_demo(kind):
    join_sql = f"""
SELECT COUNT(DISTINCT p.person_id) AS people,
       COUNT(*)                    AS result_rows
FROM person AS p
{"INNER JOIN" if kind == "INNER JOIN" else "LEFT JOIN"} condition_occurrence AS co
  ON p.person_id = co.person_id"""
    if kind.startswith("Anti"):
        join_sql += "\nWHERE co.person_id IS NULL"
    r = run(join_sql)
    display(join_toy_svg(kind), show_sql(join_sql),
            Markdown(f"**{int(r.people[0]):,}** distinct people · **{int(r.result_rows[0]):,}** rows returned\n\n"
                     "An inner join silently drops people with no events. The anti join finds exactly those people."))

## GROUP BY does aggregation

- `COUNT(*)` counts records
- `COUNT(DISTINCT person_id)` counts patients

In EHR data these differ a lot.

In [ ]:
@w.interact(
    domain=w.RadioButtons(options=[("Conditions", ("condition_occurrence", "condition_concept_id")),
                                   ("Drugs", ("drug_exposure", "drug_concept_id")),
                                   ("Visits", ("visit_occurrence", "visit_concept_id"))],
                          description="Event table"),
    count_what=w.RadioButtons(options=["COUNT(*)", "COUNT(DISTINCT e.person_id)"],
                              value="COUNT(DISTINCT e.person_id)", description="Count"),
)
def group_demo(domain, count_what):
    tbl, col = domain
    group_sql = f"""
SELECT c.concept_name, {count_what} AS n
FROM {tbl} AS e
JOIN concept AS c ON e.{col} = c.concept_id
GROUP BY c.concept_name
ORDER BY n DESC
"""
    g = run(group_sql)
    chart = alt.Chart(g).mark_bar(color=CLAUSE["group"], cornerRadiusEnd=3).encode(
        x=alt.X("n:Q", title="patients" if "DISTINCT" in count_what else "records"),
        y=alt.Y("concept_name:N", sort="-x", title=None),
        tooltip=["concept_name", alt.Tooltip("n:Q", format=",")],
    )
    display(show_sql(group_sql), styled(chart, w=520, h=min(28 * len(g), 260)))

## HAVING filters

- `WHERE` runs before grouping, on rows
- `HAVING` runs after grouping, on aggregates

Use this pattern for *high utilisers*, *patients with ≥ 2 diagnosis codes* and similar phenotype rules.

In [ ]:
_visit_dist = run("SELECT person_id, COUNT(*) AS n_visits FROM visit_occurrence GROUP BY person_id")


@w.interact(min_visits=w.IntSlider(value=8, min=1, max=15, description="Minimum visits",
                                   continuous_update=False, style={"description_width": "initial"}))
def having_demo(min_visits):
    having_sql = f"""
SELECT person_id, COUNT(*) AS n_visits
FROM visit_occurrence
GROUP BY person_id
HAVING COUNT(*) >= {min_visits}
"""
    h = run(having_sql)
    dist = _visit_dist.assign(kept=np.where(_visit_dist.n_visits >= min_visits, "kept by HAVING", "dropped"))
    hist = alt.Chart(dist).mark_bar().encode(
        x=alt.X("n_visits:O", title="visits per person"),
        y=alt.Y("count():Q", title="people"),
        color=alt.Color("kept:N", scale=alt.Scale(domain=["kept by HAVING", "dropped"],
                                                   range=[CLAUSE["having"], "#DCDDDC"]),
                        legend=alt.Legend(orient="top", title=None)),
    )
    display(show_sql(having_sql), Markdown(f"**{len(h):,}** people kept"), styled(hist, w=560, h=190))

## Dates: visits over time

**DuckDB** `date_trunc('year', d)` · **SQL Server** `DATEFROMPARTS(YEAR(d),1,1)` or `DATETRUNC(year, d)`

In [ ]:
@w.interact(
    visit_types=w.SelectMultiple(options=[("Inpatient", 9201), ("Outpatient", 9202), ("Emergency", 9203)],
                                 value=(9201, 9203), description="Visit types"),
    grain=w.ToggleButtons(options=["year", "quarter"], description="Grain"),
)
def dates_demo(visit_types, grain):
    if not visit_types:
        display(Markdown("*Pick at least one visit type.*"))
        return
    ids = ", ".join(str(v) for v in visit_types)
    dates_sql = f"""
SELECT date_trunc('{grain}', v.visit_start_date) AS period,
       c.concept_name AS visit_type,
       COUNT(*) AS n_visits
FROM visit_occurrence AS v
JOIN concept AS c ON v.visit_concept_id = c.concept_id
WHERE v.visit_concept_id IN ({ids})
GROUP BY 1, 2
ORDER BY 1
"""
    d = run(dates_sql)
    line = alt.Chart(d).mark_line(point=True, strokeWidth=2.5).encode(
        x=alt.X("period:T", title=None),
        y=alt.Y("n_visits:Q", title="visits"),
        color=alt.Color("visit_type:N", title=None, legend=alt.Legend(orient="top"),
                        scale=alt.Scale(range=[CLAUSE["order"], CLAUSE["select"], CLAUSE["where"]])),
        tooltip=["period:T", "visit_type", "n_visits"],
    )
    display(show_sql(dates_sql), styled(line, w=600, h=200))

## Measurements and CASE WHEN

`CASE WHEN` creates a derived column row by row, which is how most lab-based phenotype flags are built.

> 🟠 **Jupyter note:** the slider's range has to change when the lab changes. `interact` can't do that, so this cell wires the widgets together by hand with **`.observe()` callbacks** and an `Output` widget. In marimo, the slider is simply a cell that depends on `lab_pick`.

In [ ]:
LABS = {"HbA1c (%)": (3004410, 4.0, 13.0, 6.5),
        "Creatinine (mg/dL)": (3016723, 0.3, 4.0, 1.3),
        "Systolic BP (mmHg)": (3004249, 80.0, 190.0, 140.0)}
lab_pick = w.Dropdown(options=list(LABS), description="Lab")
cutoff = w.FloatSlider(value=6.5, min=4.0, max=13.0, step=0.1, description="Abnormal if ≥",
                       continuous_update=False, style={"description_width": "initial"})
lab_out = w.Output()


def draw_lab(*_):
    cid = LABS[lab_pick.value][0]
    lab_sql = f"""
SELECT value_as_number,
       CASE WHEN value_as_number >= {cutoff.value}
            THEN 'abnormal' ELSE 'normal' END AS flag
FROM measurement
WHERE measurement_concept_id = {cid}
  AND value_as_number IS NOT NULL
"""
    m = run(lab_sql)
    pct = (m.flag == "abnormal").mean() * 100 if len(m) else 0
    h = alt.Chart(m).mark_bar().encode(
        x=alt.X("value_as_number:Q", bin=alt.Bin(maxbins=40), title=None),
        y=alt.Y("count():Q", title="results"),
        color=alt.Color("flag:N", scale=alt.Scale(domain=["abnormal", "normal"],
                                                   range=[CLAUSE["order"], "#B3BCE3"]),
                        legend=alt.Legend(orient="top", title=None)),
    )
    rule = alt.Chart(pd.DataFrame({"x": [cutoff.value]})).mark_rule(
        color=CLAUSE["ink"], strokeDash=[5, 4], strokeWidth=2).encode(x="x:Q")
    with lab_out:
        lab_out.clear_output(wait=True)
        display(show_sql(lab_sql),
                Markdown(f"**{pct:.1f}%** of {len(m):,} results flagged abnormal"),
                styled(h + rule, w=580, h=180))


def reset_slider(change):
    _, lo, hi, default = LABS[change["new"]]
    with cutoff.hold_trait_notifications():   # avoid min > max errors mid-update
        cutoff.min, cutoff.max = min(lo, cutoff.min), max(hi, cutoff.max)
        cutoff.min, cutoff.max = lo, hi
        cutoff.step = 0.1 if hi < 20 else 1
        cutoff.value = default
    draw_lab()


lab_pick.observe(reset_slider, names="value")
cutoff.observe(draw_lab, names="value")
draw_lab()
display(w.HBox([lab_pick, cutoff]), lab_out)

## Click the chart and query the selection

Age bands come from a `CASE` expression, then `GROUP BY`. **Click a bar** (shift-click for several) to filter the table.

> 🟠 **Jupyter note:** Altair 5 can send chart selections back to Python through **`alt.JupyterChart`** (it needs the `anywidget` package). You still connect the selection to the table with an `observe` callback. In marimo, `mo.ui.altair_chart(...).value` is a DataFrame that any other cell can read.

In [ ]:
_bands = run("""
SELECT CASE
         WHEN 2024 - year_of_birth < 40 THEN '18–39'
         WHEN 2024 - year_of_birth < 65 THEN '40–64'
         ELSE '65+'
       END AS age_band,
       COUNT(*) AS people
FROM person
GROUP BY 1
ORDER BY 1
""")
pick = alt.selection_point(fields=["age_band"], name="pick")
age_chart = alt.JupyterChart(styled(
    alt.Chart(_bands).mark_bar(color=CLAUSE["select"]).encode(
        x=alt.X("age_band:N", title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("people:Q", title="people"),
        opacity=alt.condition(pick, alt.value(1), alt.value(0.35)),
    ).add_params(pick), w=360, h=240))
band_out = w.Output()


def draw_bands(*_):
    sel = age_chart.selections.pick.value or []
    picked = sorted({d["age_band"] for d in sel}) or ["18–39", "40–64", "65+"]
    in_list = ", ".join(f"'{b}'" for b in picked)
    band_sql = f"""
WITH banded AS (
  SELECT person_id,
         CASE WHEN 2024 - year_of_birth < 40 THEN '18–39'
              WHEN 2024 - year_of_birth < 65 THEN '40–64'
              ELSE '65+' END AS age_band
  FROM person
)
SELECT c.concept_name AS condition,
       COUNT(DISTINCT b.person_id) AS patients
FROM banded AS b
JOIN condition_occurrence AS co ON b.person_id = co.person_id
JOIN concept AS c ON co.condition_concept_id = c.concept_id
WHERE b.age_band IN ({in_list})
GROUP BY 1
ORDER BY 2 DESC
LIMIT 5
"""
    with band_out:
        band_out.clear_output(wait=True)
        display(Markdown(f"Showing: **{', '.join(picked)}**"), run(band_sql), show_sql(band_sql))


age_chart.selections.observe(draw_bands, ["pick"])
draw_bands()
display(w.HBox([age_chart, band_out]))

## CTEs: build a cohort step by step

*Of patients with type 2 diabetes, how many started metformin within N days of diagnosis?*
Each `WITH` block is a named step; read them top to bottom.

In [ ]:
@w.interact(window_days=w.IntSlider(value=90, min=0, max=240, step=15, description="Window (days)",
                                    continuous_update=False, style={"description_width": "initial"}))
def cohort_demo(window_days):
    cohort_sql = f"""
WITH t2dm AS (          -- index date = first T2DM diagnosis
  SELECT person_id, MIN(condition_start_date) AS index_date
  FROM condition_occurrence
  WHERE condition_concept_id = 201826
  GROUP BY person_id
),
metformin AS (          -- first metformin exposure
  SELECT person_id, MIN(drug_exposure_start_date) AS first_rx
  FROM drug_exposure
  WHERE drug_concept_id = 1503297
  GROUP BY person_id
)
SELECT COUNT(*)                                  AS t2dm_patients,
       COUNT(m.person_id)                        AS any_metformin,
       COUNT(CASE WHEN date_diff('day', t.index_date, m.first_rx)
                       BETWEEN 0 AND {window_days}
                  THEN 1 END)                    AS within_window
FROM t2dm AS t
LEFT JOIN metformin AS m ON t.person_id = m.person_id
"""
    c = run(cohort_sql).melt(var_name="step", value_name="patients")
    c["label"] = ["T2DM", "Any metformin", f"Metformin ≤ {window_days} d"]
    f = alt.Chart(c).mark_bar(color=CLAUSE["join"], cornerRadiusEnd=3).encode(
        y=alt.Y("label:N", sort=None, title=None),
        x=alt.X("patients:Q", title="patients"),
    )
    t = f.mark_text(align="left", dx=4, color=CLAUSE["ink"]).encode(text="patients:Q")
    display(styled(f + t, w=520, h=110), show_sql(cohort_sql))

---
# Beyond the basics
Pattern matching, subqueries, set operations and window functions

## LIKE finds concepts by text

- `%` matches any run of characters; `_` matches exactly one
- Use `LIKE` to *find* concept IDs, then filter event tables on the IDs
- Try `%disease`, `_etformin`, `%in%`, `%blood%`

In [ ]:
@w.interact(
    pat=w.Text(value="%chronic%", description="Pattern", continuous_update=False),
    ci=w.Checkbox(value=True, description="Case-insensitive (ILIKE)"),
    neg=w.Checkbox(value=False, description="NOT"),
)
def like_demo(pat, ci, neg):
    op = ("NOT " if neg else "") + ("ILIKE" if ci else "LIKE")
    like_sql = f"""
SELECT concept_id, concept_name, domain_id, vocabulary_id
FROM concept
WHERE concept_name {op} '{pat.replace("'", "''")}'
ORDER BY domain_id, concept_name
"""
    l = run(like_sql)
    display(show_sql(like_sql),
            Markdown(f"**{len(l):,}** of {run('SELECT COUNT(*) n FROM concept').n[0]:,} concepts match"), l)

## String functions: CONCAT, SPLIT_PART, REGEXP_EXTRACT

- `||` concatenates
- `SPLIT_PART` and `regexp_extract` pull pieces out of a string
- Source values in OMOP (`*_source_value`) are free text, so these functions do most of the cleanup

In [ ]:
@w.interact(domain=w.ToggleButtons(options=["Measurement", "Drug", "Condition"], description="Domain"))
def str_demo(domain):
    str_sql = f"""
SELECT concept_name,
       LOWER(concept_name)                   AS lower_name,
       LENGTH(concept_name)                  AS n_chars,
       vocabulary_id || ':' || concept_id    AS code_label,
       SPLIT_PART(concept_name, ' ', 1)      AS first_word,
       regexp_extract(concept_name, '\\[(.*)\\]', 1) AS in_brackets
FROM concept
WHERE domain_id = '{domain}'
"""
    display(show_sql(str_sql), run(str_sql))

## Subqueries: IN, EXISTS and NOT EXISTS

*Which patients have a condition but no record of the usual drug?* This is a classic care-gap query.

- `EXISTS` stops at the first match, so it never duplicates rows the way a `JOIN` can
- `NOT IN` breaks if the subquery returns a `NULL`; prefer `NOT EXISTS`

In [ ]:
@w.interact(
    cond=w.Dropdown(options=[("Type 2 diabetes", 201826), ("Essential hypertension", 320128),
                             ("Coronary arteriosclerosis", 317576)], description="Has condition",
                    style={"description_width": "initial"}),
    drug=w.Dropdown(options=[("metformin", 1503297), ("lisinopril", 1308216), ("amlodipine", 1332418),
                             ("atorvastatin", 1545958), ("aspirin", 1112807)], description="Drug"),
    mode=w.ToggleButtons(options=["EXISTS", "NOT EXISTS"], value="NOT EXISTS", description="Drug check"),
)
def sub_demo(cond, drug, mode):
    sub_sql = f"""
SELECT COUNT(*) AS patients
FROM person AS p
WHERE p.person_id IN (                 -- has the condition
        SELECT person_id FROM condition_occurrence
        WHERE condition_concept_id = {cond})
  AND {mode} (                   -- correlated subquery
        SELECT 1 FROM drug_exposure AS d
        WHERE d.person_id = p.person_id
          AND d.drug_concept_id = {drug})
"""
    n = int(run(sub_sql).patients[0])
    base = int(run(f"SELECT COUNT(DISTINCT person_id) n FROM condition_occurrence "
                   f"WHERE condition_concept_id = {cond}").n[0])
    display(show_sql(sub_sql),
            Markdown(f"**{base:,}** with the condition · **{n:,}** "
                     f"{'on' if mode == 'EXISTS' else 'not on'} the drug · "
                     f"**{100 * n / base:.0f}%**" if base else "–"))

## UNION, INTERSECT and EXCEPT

Set operators stack two queries with the same columns. `UNION`, `INTERSECT` and `EXCEPT` remove duplicates; `UNION ALL` keeps them, so a patient with both conditions appears twice.

In [ ]:
_CONDS = [("Type 2 diabetes", 201826), ("Essential hypertension", 320128), ("Obesity", 433736),
          ("Chronic kidney disease", 46271022), ("Depressive disorder", 440383)]


@w.interact(a=w.Dropdown(options=_CONDS, value=201826, description="A"),
            b=w.Dropdown(options=_CONDS, value=320128, description="B"),
            op=w.ToggleButtons(options=["UNION", "INTERSECT", "EXCEPT", "UNION ALL"], value="INTERSECT",
                               description="Operator"))
def set_demo(a, b, op):
    ab = run(f"""
SELECT COUNT(DISTINCT person_id) FILTER (WHERE condition_concept_id = {a}) AS a,
       COUNT(DISTINCT person_id) FILTER (WHERE condition_concept_id = {b}) AS b
FROM condition_occurrence""")
    both = int(run(f"""
SELECT COUNT(*) AS n FROM (
  SELECT person_id FROM condition_occurrence WHERE condition_concept_id = {a}
  INTERSECT
  SELECT person_id FROM condition_occurrence WHERE condition_concept_id = {b})""").n[0])
    na, nb = int(ab.a[0]), int(ab.b[0])
    fill = CLAUSE["from"]
    shade = {
        "UNION": f'<circle cx="160" cy="130" r="100" fill="{fill}"/><circle cx="260" cy="130" r="100" fill="{fill}"/>',
        "UNION ALL": f'<circle cx="160" cy="130" r="100" fill="{fill}"/><circle cx="260" cy="130" r="100" fill="{fill}"/>',
        "INTERSECT": f'<circle cx="160" cy="130" r="100" fill="{fill}" clip-path="url(#venn-b)"/>',
        "EXCEPT": f'<circle cx="160" cy="130" r="100" fill="{fill}" mask="url(#venn-not-b)"/>',
    }[op]
    set_sql = f"""
SELECT person_id FROM condition_occurrence
WHERE condition_concept_id = {a}      -- A
{op}
SELECT person_id FROM condition_occurrence
WHERE condition_concept_id = {b}      -- B
"""
    s = run(set_sql)
    display(HTML(f"""
<svg viewBox="0 0 420 260" width="100%" style="max-width:420px" role="img" aria-label="Venn diagram of the two patient sets">
  <style>.v{{font:700 20px 'JetBrains Mono',monospace}}.vl{{font:15px Georgia,serif;fill:#5F6168}}</style>
  <defs>
    <clipPath id="venn-b"><circle cx="260" cy="130" r="100"/></clipPath>
    <mask id="venn-not-b"><rect width="420" height="260" fill="#fff"/><circle cx="260" cy="130" r="100" fill="#000"/></mask>
  </defs>
  <g opacity="0.3">{shade}</g>
  <circle cx="160" cy="130" r="100" fill="none" stroke="{fill}" stroke-width="2.5"/>
  <circle cx="260" cy="130" r="100" fill="none" stroke="{fill}" stroke-width="2.5"/>
  <text class="vl" x="70" y="22">A</text><text class="vl" x="340" y="22">B</text>
  <text class="v" x="105" y="137" text-anchor="middle">{na - both}</text>
  <text class="v" x="210" y="137" text-anchor="middle">{both}</text>
  <text class="v" x="315" y="137" text-anchor="middle">{nb - both}</text>
  <text class="vl" x="210" y="252" text-anchor="middle">distinct patients in each region</text>
</svg>"""), show_sql(set_sql),
            Markdown(f"**{len(s):,}** rows returned, **{s.person_id.nunique():,}** distinct patients"))

## NULL-safe summaries: COALESCE, NULLIF, FILTER

- `race_concept_id = 0` means “no matching concept”, so the `LEFT JOIN` returns `NULL`; `COALESCE` swaps in a label
- `FILTER (WHERE …)` counts a subset inside one group, turning rows into columns
- `NULLIF(x, 0)` returns `NULL` when `x` is 0, so a division gives `NULL` instead of an error

In [ ]:
@w.interact(use_coalesce=w.Checkbox(value=True, description="Wrap in COALESCE"))
def null_demo(use_coalesce):
    race = "COALESCE(c.concept_name, 'Unknown')" if use_coalesce else "c.concept_name"
    null_sql = f"""
SELECT {race} AS race,
       COUNT(*) AS people,
       COUNT(*) FILTER (WHERE p.gender_concept_id = 8532) AS female,
       COUNT(*) FILTER (WHERE p.gender_concept_id = 8507) AS male,
       ROUND(100.0 * COUNT(*) FILTER (WHERE p.gender_concept_id = 8532)
             / NULLIF(COUNT(*), 0), 1) AS pct_female
FROM person AS p
LEFT JOIN concept AS c ON p.race_concept_id = c.concept_id
GROUP BY 1
ORDER BY people DESC
"""
    display(show_sql(null_sql), run(null_sql))

## OVER aggregates without collapsing rows

`GROUP BY` turns 3 visits into 1 row. A window function keeps all 3 rows and adds the aggregate to each one.

In [ ]:
@w.interact(mode=w.RadioButtons(options=["GROUP BY", "Window: OVER (PARTITION BY …)"],
                                value="Window: OVER (PARTITION BY …)", description="Show"))
def win_demo(mode):
    if mode == "GROUP BY":
        win_sql = """
SELECT person_id,
       COUNT(*)              AS n_visits,
       MIN(visit_start_date) AS first_visit
FROM visit_occurrence
WHERE person_id IN (1, 2, 3)
GROUP BY person_id
ORDER BY person_id"""
    else:
        win_sql = """
SELECT person_id, visit_start_date,
       COUNT(*)              OVER (PARTITION BY person_id) AS n_visits,
       MIN(visit_start_date) OVER (PARTITION BY person_id) AS first_visit,
       date_diff('day',
         MIN(visit_start_date) OVER (PARTITION BY person_id),
         visit_start_date)                                AS days_since_first
FROM visit_occurrence
WHERE person_id IN (1, 2, 3)
ORDER BY person_id, visit_start_date"""
    display(show_sql(win_sql), run(win_sql))

## ROW_NUMBER, RANK and DENSE_RANK

- On ties, `ROW_NUMBER` still counts 1, 2, 3, `RANK` repeats and then skips, and `DENSE_RANK` repeats without skipping
- `ROW_NUMBER() … = 1` is the standard OMOP “first event per person” pattern
- DuckDB also lets you write `QUALIFY rn = 1` instead of the outer query

In [ ]:
@w.interact(person=w.Dropdown(options=["85", "306", "679", "1005"], value="306", description="Patient"),
            first_only=w.Checkbox(value=False, description="First diagnosis per patient (rn = 1)"))
def rank_demo(person, first_only):
    if not first_only:
        rank_sql = f"""
SELECT co.condition_start_date, c.concept_name,
       ROW_NUMBER() OVER w AS row_num,
       RANK()       OVER w AS rnk,
       DENSE_RANK() OVER w AS dense_rnk
FROM condition_occurrence AS co
JOIN concept AS c ON co.condition_concept_id = c.concept_id
WHERE co.person_id = {person}
WINDOW w AS (ORDER BY co.condition_start_date)
ORDER BY row_num"""
    else:
        rank_sql = """
WITH ranked AS (
  SELECT co.person_id, c.concept_name, co.condition_start_date,
         ROW_NUMBER() OVER (PARTITION BY co.person_id
                            ORDER BY co.condition_start_date,
                                     co.condition_occurrence_id) AS rn
  FROM condition_occurrence AS co
  JOIN concept AS c ON co.condition_concept_id = c.concept_id
)
SELECT person_id, concept_name AS first_condition, condition_start_date
FROM ranked
WHERE rn = 1
ORDER BY person_id"""
    rk = run(rank_sql)
    display(show_sql(rank_sql), rk.head(10))

## LEAD and LAG: time between visits

- `LEAD(x)` reads the next row's value in the window and `LAG(x)` reads the previous one
- This is how readmission and gap-in-care measures are built

In [ ]:
@w.interact(
    lag_types=w.SelectMultiple(options=[("Inpatient", 9201), ("Outpatient", 9202), ("Emergency", 9203)],
                               value=(9201,), description="Index visits"),
    return_days=w.IntSlider(value=30, min=7, max=180, description="Return within (days)",
                            continuous_update=False, style={"description_width": "initial"}),
)
def lag_demo(lag_types, return_days):
    if not lag_types:
        display(Markdown("*Pick at least one visit type.*"))
        return
    ids = ", ".join(str(v) for v in lag_types)
    lag_sql = f"""
WITH seq AS (
  SELECT person_id, visit_concept_id, visit_end_date,
         LEAD(visit_start_date) OVER (PARTITION BY person_id
                                      ORDER BY visit_start_date) AS next_visit
  FROM visit_occurrence
)
SELECT date_diff('day', visit_end_date, next_visit) AS days_to_next
FROM seq
WHERE visit_concept_id IN ({ids})
  AND next_visit >= visit_end_date
"""
    g = run(lag_sql)
    g["returned"] = np.where(g.days_to_next <= return_days, f"within {return_days} d", "later")
    pct = (g.days_to_next <= return_days).mean() * 100 if len(g) else 0
    hist = alt.Chart(g[g.days_to_next <= 730]).mark_bar().encode(
        x=alt.X("days_to_next:Q", bin=alt.Bin(step=15), title="days to next visit (first 2 years)"),
        y=alt.Y("count():Q", title="visits"),
        color=alt.Color("returned:N", sort="descending",
                        scale=alt.Scale(range=[CLAUSE["window"], "#DCDDDC"]),
                        legend=alt.Legend(orient="top", title=None)),
    )
    display(show_sql(lag_sql),
            Markdown(f"**{pct:.1f}%** of {len(g):,} index visits had another visit within {return_days} days"),
            styled(hist, w=580, h=170))

## Window frames: running and moving values

- The frame says which rows around the current one feed the aggregate
- `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` gives a 3-reading moving average
- With only `ORDER BY`, the frame runs from the first row, giving a running total or max

In [ ]:
@w.interact(person=w.Dropdown(options=["806", "648", "644", "431"], value="806", description="Patient"),
            k=w.IntSlider(value=2, min=0, max=5, description="ROWS … PRECEDING",
                          style={"description_width": "initial"}))
def roll_demo(person, k):
    roll_sql = f"""
SELECT measurement_date,
       value_as_number AS sbp,
       AVG(value_as_number) OVER (PARTITION BY person_id ORDER BY measurement_date
             ROWS BETWEEN {k} PRECEDING AND CURRENT ROW) AS moving_avg,
       MAX(value_as_number) OVER (PARTITION BY person_id
             ORDER BY measurement_date)                   AS running_max
FROM measurement
WHERE measurement_concept_id = 3004249        -- systolic BP
  AND person_id = {person}
ORDER BY measurement_date
"""
    r = run(roll_sql).melt(id_vars="measurement_date", var_name="series", value_name="mmHg")
    ln = alt.Chart(r).mark_line(point=True).encode(
        x=alt.X("measurement_date:T", title=None),
        y=alt.Y("mmHg:Q", scale=alt.Scale(zero=False), title="systolic BP (mmHg)"),
        color=alt.Color("series:N", title=None, legend=alt.Legend(orient="top"),
                        scale=alt.Scale(domain=["sbp", "moving_avg", "running_max"],
                                        range=["#B3BCE3", CLAUSE["window"], CLAUSE["ink"]])),
        strokeDash=alt.StrokeDash("series:N", legend=None,
                                  scale=alt.Scale(domain=["sbp", "moving_avg", "running_max"],
                                                  range=[[1, 0], [1, 0], [5, 4]])),
        tooltip=["measurement_date:T", "series", alt.Tooltip("mmHg:Q", format=".1f")],
    )
    display(show_sql(roll_sql), styled(ln, w=600, h=190))

---
## 🟠 Hidden state, demonstrated

This is the behaviour marimo was designed to remove. Run the three cells below **in order**. Then:

1. Change `threshold` in **cell A** to `8.0` and re-run **only cell A**
2. Re-run **only cell C**

Cell C now prints the **new** threshold next to the **old** count, because cell B never reran. Nothing warns you. The execution counts (`[n]`) on the left are the only clue.

In marimo, editing cell A would rerun B and C automatically, so they can't fall out of step.

In [ ]:
# cell A
threshold = 6.5

In [ ]:
# cell B
n_flagged = int(run(f"""
SELECT COUNT(*) AS n FROM measurement
WHERE measurement_concept_id = 3004410 AND value_as_number >= {threshold}
""").n[0])

In [ ]:
# cell C
print(f"{n_flagged:,} HbA1c results are ≥ {threshold}%")

> **One more experiment:** delete cell A completely and re-run cell C. It still works, because `threshold` lives on in the kernel's memory. Restart the kernel and **Run All**, and the notebook breaks.
> marimo deletes a cell's variables when you delete the cell, so the notebook you see always matches the program that runs.

## Exercise

Write any query against `person`, `visit_occurrence`, `condition_occurrence`, `drug_exposure`, `measurement` or `concept`, then press **Run query**.

In [ ]:
editor = w.Textarea(
    value="""SELECT c.concept_name, ROUND(AVG(m.value_as_number), 2) AS mean_value
FROM measurement AS m
JOIN concept AS c ON m.measurement_concept_id = c.concept_id
GROUP BY c.concept_name""",
    layout=w.Layout(width="100%", height="140px", font_family="monospace"))
run_btn = w.Button(description="Run query", button_style="primary")
query_out = w.Output()


def on_run(_):
    with query_out:
        query_out.clear_output()
        try:
            out = run(editor.value)
            display(Markdown(f"{len(out):,} rows"), out.head(20))
        except Exception as e:
            display(Markdown(f"**The query failed.** {e}"))


run_btn.on_click(on_run)
display(editor, run_btn, query_out)

## Same query, three ways

| Task | DuckDB (this notebook) | SQL Server | PostgreSQL |
|---|---|---|---|
| First *n* rows | `LIMIT 10` | `SELECT TOP 10 …` | `LIMIT 10` |
| Days between dates | `date_diff('day', a, b)` | `DATEDIFF(day, a, b)` | `b - a` |
| Truncate to year | `date_trunc('year', d)` | `DATETRUNC(year, d)` | `date_trunc('year', d)` |
| Current date | `current_date` | `CAST(GETDATE() AS date)` | `current_date` |
| String concat | `a \|\| b` | `CONCAT(a, b)` | `a \|\| b` |
| Case-insensitive match | `s ILIKE '%ckd%'` | `s LIKE '%ckd%'` (default collation ignores case) | `s ILIKE '%ckd%'` |
| Regex match | `regexp_matches(s, 'p')` | `LIKE` patterns only (`REGEXP_LIKE` from 2025) | `s ~ 'p'` |
| Conditional count | `COUNT(*) FILTER (WHERE …)` | `SUM(CASE WHEN … THEN 1 ELSE 0 END)` | `COUNT(*) FILTER (WHERE …)` |
| Keep first row per group | `QUALIFY ROW_NUMBER() OVER (…) = 1` | subquery with `WHERE rn = 1` | subquery with `WHERE rn = 1` |
| Table reference | `cdm.person` | `cdm_db.dbo.person` | `cdm.person` |

OHDSI’s **SqlRender** translates one parameterised query into each dialect, which is how ATLAS and HADES packages stay portable.

## Export the result

Pipe-delimited with every field quoted survives commas inside concept names. The year leads the filename so extracts sort by period.

> 🟠 **Jupyter note:** ipywidgets has no download button, so the link below embeds the file as a base64 `data:` URI. In marimo this is `mo.download(...)`.

In [ ]:
@w.interact(year=w.Dropdown(options=[str(y) for y in range(2016, 2025)], value="2024",
                            description="Diagnosis year", style={"description_width": "initial"}))
def export_demo(year):
    x = run(f"""
SELECT co.person_id, c.concept_name, co.condition_start_date
FROM condition_occurrence AS co
JOIN concept AS c ON co.condition_concept_id = c.concept_id
WHERE co.condition_concept_id = 201826
  AND EXTRACT(year FROM co.condition_start_date) = {year}
ORDER BY co.condition_start_date
""")
    txt = x.to_csv(sep="|", index=False, quoting=csv.QUOTE_ALL)
    fname = f"{year}_t2dm_cohort.tsv"
    b64 = base64.b64encode(txt.encode()).decode()
    display(Markdown(f"**{len(x):,}** rows → `{fname}`"),
            Markdown("```text\n" + "\n".join(txt.splitlines()[:5]) + "\n```"),
            HTML(f'<a download="{fname}" href="data:text/tab-separated-values;base64,{b64}">⬇ Download {fname}</a>'))

---
## Jupyter vs. marimo vs. Quarto

| | Jupyter (`.ipynb`, this file) | Quarto + marimo engine (`.qmd`) | Native marimo (`.py`) |
|---|---|---|---|
| **File format** | JSON with outputs embedded | Markdown with ` ```{python .marimo} ` blocks | Plain Python: script, app or notebook |
| **Execution model** | You run cells in whatever order you like; state persists | Reactive (marimo), rendered into slides | Reactive: change a cell and its dependents rerun |
| **Hidden state** | Common (see the demo above) | None | None |
| **Interactivity** | `interact`, `observe` callbacks, `Output` widgets | `mo.ui` widgets in the browser (Pyodide) | `mo.ui` widgets, table and chart selections, forms |
| **Cross-cell widgets** | Manual: one cell, or callbacks | Automatic | Automatic |
| **Git diffs** | Noisy (outputs and metadata) | Clean | Clean |
| **Sharing** | nbviewer, GitHub preview, Colab, Binder | Static site | molab link, badge, WASM embed |

**Open this notebook elsewhere**

- **Colab:** `https://colab.research.google.com/github/OWNER/REPO/blob/main/sql-basics-jupyter.ipynb`
- **Convert to marimo:** `uvx marimo convert sql-basics-jupyter.ipynb -o converted.py`. The `interact` cells convert, but they keep their callback style. Compare the result with `sql-basics-marimo.py`.

## Recap

| Clause | Job |
|---|---|
| **FROM** | which table (and alias) |
| **JOIN … ON** | link tables, usually on `person_id` or `*_concept_id` |
| **WHERE** | keep rows, before any grouping |
| **GROUP BY** | collapse rows into groups for `COUNT`, `AVG`, `MIN` |
| **HAVING** | keep groups, after aggregation |
| **SELECT** | choose and compute output columns, incl. `CASE WHEN` |
| **ORDER BY** | sort, then `LIMIT` to trim |
| **LIKE / ILIKE** | match text with `%` and `_`, to find concept IDs |
| **EXISTS** | keep rows that have (or lack) a related row, without duplicating them |
| **UNION · INTERSECT · EXCEPT** | combine two result sets that have the same columns |
| **COALESCE · NULLIF · FILTER** | replace `NULL`s, prevent division by zero, count a subset per group |
| **OVER · ROW_NUMBER · LAG/LEAD · ROWS BETWEEN** | per-group aggregates that keep every row, first events, gaps, running values |

Built with [DuckDB](https://duckdb.org/), [Quarto](https://quarto.org/) and [marimo](https://marimo.io/gallery).